# E02: Programación Orientada a Objetos en Python

## Objetivos

Al finalizar este módulo serás capaz de:

1. **Crear y utilizar clases** con `__init__`, `self`, atributos de instancia, métodos y type hints correctos.
2. **Implementar encapsulamiento** usando `@property`, convenciones de naming (`_var`, `__var`) y name mangling.
3. **Distinguir `@classmethod` de `@staticmethod`** y aplicar factory methods cuando sea apropiado.
4. **Dominar la herencia** simple y múltiple, incluyendo `super()`, `isinstance`, `issubclass` y la resolución MRO (C3 linearization).
5. **Evaluar cuándo usar composición vs herencia** y diseñar sistemas que favorezcan la composición.

## Analogía inicial: La fábrica y el piloto

### Clases = Plantillas de fabricación

Piensa en una **fábrica de automóviles**. La planta tiene un **plano** (la clase) que define cómo se construye cada coche: motor, ruedas, color, capacidad. Ese plano no es un coche concreto — es un **molde**.

Cuando la fábrica produce un vehículo, obtienes un **objeto** concreto: un Toyota rojo, un Mazda azul. Cada uno tiene sus propios valores, pero todos comparten la misma estructura.

```
   PLANO (Clase Auto)              OBJETOS (Instancias)
  ┌─────────────────┐         ┌──────────┐ ┌──────────┐
  │ • marca: str     │         │ Toyota   │ │ Mazda    │
  │ • color: str     │──────►  │ rojo     │ │ azul     │
  │ • arrancar()     │         │ arrancar │ │ arrancar │
  └─────────────────┘         └──────────┘ └──────────┘
      FÁBRICA                    Producto 1  Producto 2
```

### Encapsulamiento = Cabina de vuelo

Un avión tiene controles críticos **encerrados** en la cabina. El piloto accede a ellos mediante instrumentos (propiedades) pero no puede modificar directamente el cableado interno del motor. Si algo falla, hay una barrera de protección.

En Python, el encapsulamiento funciona igual: los atributos "privados" están protegidos y solo se acceden a través de métodos controlados (`@property`).

## 1. Fundamentos de Clases y Objetos

### 1.1 Primera clase: `class`, `__init__` y `self`

Una clase se define con la palabra clave `class`. El método `__init__` es el **constructor** — se ejecuta automáticamente cuando creas una instancia. El parámetro `self` referencia al objeto que se está creando.

In [ ]:
class Vehiculo:
    """Clase base que representa un vehículo genérico."""

    def __init__(self, marca: str, color: str, anio: int) -> None:
        self.marca = marca
        self.color = color
        self.anio = anio
        self._velocidad = 0  #atributo "protegido" (convención)

    def arrancar(self) -> str:
        """Simula arrancar el vehículo."""
        return f"{self.marca} ha arrancado."

    def acelerar(self, incremento: int) -> int:
        """Aumenta la velocidad actual."""
        self._velocidad += incremento
        return self._velocidad

    def frenar(self, decremento: int) -> int:
        """Reduce la velocidad actual."""
        self._velocidad = max(0, self._velocidad - decremento)
        return self._velocidad

    def __repr__(self) -> str:
        return f"Vehiculo(marca='{self.marca}', color='{self.color}', anio={self.anio})")

In [ ]:
# Crear instancias (objetos concretos)
coche1 = Vehiculo("Toyota", "Rojo", 2023)
coche2 = Vehiculo("Mazda", "Azul", 2024)

print(coche1)
print(coche1.arrancar())
print(f"Velocidad: {coche1.acelerar(60)} km/h")
print(f"Velocidad: {coche1.frenar(20)} km/h")
print()
print(coche2)
print(coche2.arrancar())

### 1.2 Atributos de clase vs instancia

Los **atributos de clase** son compartidos por todas las instancias. Los **atributos de instancia** son únicos para cada objeto.

In [ ]:
class CuentaBancaria:
    """Demuestra atributos de clase vs instancia."""

    # Atributo de clase: compartido por todas las cuentas
    tasa_interes: float = 0.05
    total_cuentas: int = 0

    def __init__(self, titular: str, saldo: float = 0.0) -> None:
        # Atributos de instancia: únicos por cada cuenta
        self.titular = titular
        self.saldo = saldo
        CuentaBancaria.total_cuentas += 1

    def aplicar_interes(self) -> float:
        """Aplica la tasa de interés al saldo."""
        self.saldo += self.saldo * CuentaBancaria.tasa_interes
        return self.saldo

    def __repr__(self) -> str:
        return f"CuentaBancaria(titular='{self.titular}', saldo={self.saldo:.2f})")

In [ ]:
c1 = CuentaBancaria("Ana", 1000)
c2 = CuentaBancaria("Luis", 5000)

print(f"Total de cuentas creadas: {CuentaBancaria.total_cuentas}")
print(f"Tasa de interés global: {CuentaBancaria.tasa_interes}")
print()
print(f"{c1} → nuevo saldo: {c1.aplicar_interes():.2f}")
print(f"{c2} → nuevo saldo: {c2.aplicar_interes():.2f}")
print(f"Total de cuentas: {CuentaBancaria.total_cuentas}")

### 1.3 Type hints en métodos

Python 3.12+ soporta type hints completos. Especificar tipos en los parámetros de retorno y argumentos mejora la legibilidad y permite herramientas como `mypy`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Producto:
    """Producto con type hints completos y dataclass."""

    nombre: str
    precio: float
    cantidad: int = 0

    def subtotal(self) -> float:
        """Calcula subtotal = precio × cantidad."""
        return self.precio * self.cantidad

    def aplicar_descuento(self, porcentaje: float) -> float:
        """Aplica un descuento porcentual y retorna el nuevo precio."""
        if not 0 <= porcentaje <= 100:
            raise ValueError(f"El descuento debe estar entre 0 y 100, recibido: {porcentaje}")
        descuento = self.precio * (porcentaje / 100)
        self.precio -= descuento
        return self.precio


p = Producto("Teclado", 45.99, 3)
print(f"Producto: {p}")
print(f"Subtotal: ${p.subtotal():.2f}")
print(f"Precio con 10% descuento: ${p.aplicar_descuento(10):.2f}")

### 1.4 Sobrecarga de métodos (o la ausencia de ella)

En lenguajes como Java o C++, puedes definir múltiples métodos con el mismo nombre pero distintos parámetros (**sobrecarga**). **Python NO soporta sobrecarga de métodos**.

Si defines el mismo nombre dos veces, la segunda definición **sobreescribe** la primera:

In [ ]:
# Esto es lo que NO se debe hacer:
class Ejemplo:
    def calcular(self, x: int) -> int:
        return x * 2

    def calcular(self, x: str) -> str:  # Sobreescribe el anterior
        return x.upper()


ej = Ejemplo()
# Solo el segundo método existe:
print(ej.calcular(10))  # TypeError: expected str, got int

In [ ]:
from typing import overload


# El patrón correcto en Python: usar typing.overload para hints
class Calculadora:
    """Usa @overload para documentar variantes, pero una sola implementación."""

    @overload
    def calcular(self, x: int) -> int: ...

    @overload
    def calcular(self, x: str) -> str: ...

    def calcular(self, x: int | str) -> int | str:
        """Implementación única con lógica condicional."""
        if isinstance(x, int):
            return x * 2
        return x.upper()


calc = Calculadora()
print(f"Entero: {calc.calcular(10)}")
print(f"String: {calc.calcular('hola')}")

### 1.5 Métodos especiales (dunder methods)

Python usa métodos con doble guion bajo (`__method__`) para integrar tus clases con operadores y funciones built-in.

In [ ]:
class Punto:
    """Representa un punto 2D con operadores intuitivos."""

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Punto({self.x}, {self.y})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Punto):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __add__(self, other: "Punto") -> "Punto":
        """Suma dos puntos: Punto(1,2) + Punto(3,4) = Punto(4,6)"""
        return Punto(self.x + other.x, self.y + other.y)

    def __mul__(self, escalar: float) -> "Punto":
        """Multiplica por un escalar: Punto(2,3) * 2 = Punto(4,6)"""
        return Punto(self.x * escalar, self.y * escalar)

    def __abs__(self) -> float:
        """Distancia al origen: abs(Punto(3,4)) = 5.0"""
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __len__(self) -> int:
        """Retorna la componente X como entero (ejemplo didáctico)."""
        return int(self.x)


a = Punto(3, 4)
b = Punto(1, 2)

print(f"Punto a: {a}")
print(f"Punto b: {b}")
print(f"a + b = {a + b}")
print(f"a * 3 = {a * 3}")
print(f"|a| = {abs(a)}")
print(f"a == b? {a == b}")
print(f"a == Punto(3,4)? {a == Punto(3, 4)}")

## 2. `@property` y Encapsulamiento

### 2.1 `@property` como getter, setter y deleter

`@property` te permite acceder a un método como si fuera un atributo, pero con lógica controlada detrás. Es la forma **pythónica** de implementar getters y setters.

In [ ]:
class Temperatura:
    """Almacena temperatura en Celsius con validación."""

    def __init__(self, celsius: float = 0.0) -> None:
        self._celsius = celsius  # atributo "protegido"

    @property
    def celsius(self) -> float:
        """Getter: retorna la temperatura en Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, valor: float) -> None:
        """Setter: valida antes de asignar."""
        if valor < -273.15:
            raise ValueError(f"Temperatura imposible: {valor}°C")
        self._celsius = valor

    @celsius.deleter
    def celsius(self) -> None:
        """Deleter: resetea a cero."""
        print("Resetando temperatura a 0°C")
        self._celsius = 0.0

    @property
    def fahrenheit(self) -> float:
        """Propiedad calculada: Celsius a Fahrenheit."""
        return self._celsius * 9 / 5 + 32

    @property
    def kelvin(self) -> float:
        """Propiedad calculada: Celsius a Kelvin."""
        return self._celsius + 273.15


t = Temperatura(37.5)
print(f"Celsius: {t.celsius}°C")
print(f"Fahrenheit: {t.fahrenheit}°F")
print(f"Kelvin: {t.kelvin}K")
print()

# Setter con validación
t.celsius = 100
print(f"Nueva temperatura: {t.celsius}°C")

# Setter con valor inválido
try:
    t.celsius = -300
except ValueError as e:
    print(f"Error: {e}")
print()

# Deleter
del t.celsius
print(f"Después de delete: {t.celsius}°C")

### 2.2 Convenciones de naming para encapsulamiento

```
┌──────────────────────────────────────────────────────────────────────┐
│  Convención        │  Ejemplo      │  Significado                   │
├────────────────────┼───────────────┼────────────────────────────────┤
│  público           │  self.nombre   │  Accesible desde cualquier lado │
│  protegido (_var)  │  self._edad   │  "No toques esto" (convención) │
│  privado (__var)   │  self.__pwd   │  Name mangling: _Clase__pwd    │
│  dunder (__var__)  │  self.__init__│  Método especial de Python     │
└──────────────────────────────────────────────────────────────────────┘
```

- **`_atributo`** (un guion bajo): es una **convención**. Python no impide el acceso, pero indica a otros desarrolladores: "esto es interno, no lo modifiques directamente".
- **`__atributo`** (doble guion bajo): activa el **name mangling**. Python renombra internamente el atributo a `_Clase__atributo`, lo que dificulta (pero no impide) el acceso externo.

In [ ]:
class CuentaSegura:
    """Demuestra name mangling con doble underscore."""

    def __init__(self, titular: str, pin: str) -> None:
        self.titular = titular      # público
        self._saldo = 0.0           # protegido (convención)
        self.__pin = pin            # name mangling: _CuentaSegura__pin

    def verificar_pin(self, pin: str) -> bool:
        return self.__pin == pin

    @property
    def saldo(self) -> float:
        """Solo lectura: no hay setter."""
        return self._saldo

    def depositar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto debe ser positivo")
        self._saldo += monto


cuenta = CuentaSegura("María", "1234")

# Acceso normal
print(f"Titular: {cuenta.titular}")
print(f"Saldo: {cuenta.saldo}")
cuenta.depositar(500)
print(f"Saldo después de depósito: {cuenta.saldo}")
print(f"PIN correcto: {cuenta.verificar_pin('1234')}")
print()

# Name mangling en acción
print("Acceso directo al PIN con name mangling:")
print(f"  cuenta.__pin → AttributeError")
print(f"  cuenta._CuentaSegura__pin → {cuenta._CuentaSegura__pin}")

## 3. `@classmethod` y `@staticmethod`

### 3.1 Diferencia clave

```
┌──────────────────────────────────────────────────────────────────────┐
│              │  Recibe          │  ¿Puede acceder a  │  Uso típico  │
│              │                  │  atributos clase?  │              │
├──────────────┼──────────────────┼────────────────────┼──────────────┤
│ @classmethod │  cls (la clase)  │  Sí                │  Factory     │
│ @staticmethod │  Ninguno        │  No                │  Utilidad    │
│ método inst. │  self (instancia)│  Sí                │  Lógica obj. │
└──────────────────────────────────────────────────────────────────────┘
```

- **`@classmethod`**: recibe `cls` (la clase, no la instancia). Útil para **factory methods** que crean objetos de distintas formas.
- **`@staticmethod`**: no recibe ni `self` ni `cls`. Es una función que vive dentro de la clase por organización, pero no accede a nada de la clase.

In [ ]:
from datetime import datetime, date


class Evento:
    """Demuestra @classmethod como factory method y @staticmethod."""

    def __init__(self, nombre: str, fecha: date) -> None:
        self.nombre = nombre
        self.fecha = fecha

    # Factory method 1: desde un objeto date
    @classmethod
    def desde_fecha(cls, nombre: str, anio: int, mes: int, dia: int) -> "Evento":
        """Crea un Evento desde componentes numéricos."""
        return cls(nombre, date(anio, mes, dia))

    # Factory method 2: desde string
    @classmethod
    def desde_string(cls, texto: str) -> "Evento":
        """Crea un Evento desde 'Nombre | 2025-03-15'."""
        nombre, fecha_str = texto.split(" | ")
        fecha = datetime.strptime(fecha_str.strip(), "%Y-%m-%d").date()
        return cls(nombre.strip(), fecha)

    # Factory method 3: evento de hoy
    @classmethod
    def de_hoy(cls, nombre: str) -> "Evento":
        """Crea un Evento con la fecha de hoy."""
        return cls(nombre, date.today())

    # Static method: función utilitaria sin acceso a la clase
    @staticmethod
    def es_finde(fecha: date) -> bool:
        """Verifica si una fecha cae en fin de semana."""
        return fecha.weekday() >= 5  # 5=sábado, 6=domingo

    def __repr__(self) -> str:
        return f"Evento('{self.nombre}', {self.fecha})"


# Usar los factory methods
e1 = Evento.desde_fecha("Conferencia", 2025, 6, 15)
e2 = Evento.desde_string("Meetup Python | 2025-09-20")
e3 = Evento.de_hoy("Sesión de estudio")

print(f"e1: {e1}")
print(f"e2: {e2}")
print(f"e3: {e3}")
print()

# Usar static method
print(f"¿{e1.fecha} es finde? {Evento.es_finde(e1.fecha)}")
print(f"¿{e2.fecha} es finde? {Evento.es_finde(e2.fecha)}")

### 3.2 ¿Cuándo usar cada uno?

**`@classmethod`** cuando:
- Necesitas crear instancias de la clase de formas alternativas (factory methods).
- Quieres acceder a atributos de clase (como contadores, configuración compartida).
- Trabajas con herencia y necesitas que `cls` sea la subclase correcta.

**`@staticmethod`** cuando:
- Tienes una función lógicamente relacionada con la clase pero no necesita acceder a `self` ni `cls`.
- Es una función de utilidad/pure math que simplemente vive en el namespace de la clase.

**Método de instancia** cuando:
- La operación depende del estado específico del objeto.

## 4. Herencia

### 4.1 Herencia simple: `extends`

La herencia permite crear una **subclase** que reutiliza y extiende el comportamiento de una **superclase**. Usa la relación **"es un"**: un `Perro` *es un* `Animal`.

In [ ]:
class Animal:
    """Superclase base."""

    def __init__(self, nombre: str, especie: str) -> None:
        self.nombre = nombre
        self.especie = especie
        self._energia = 100

    def hablar(self) -> str:
        """Método que las subclases sobreescriben (override)."""
        return f"{self.nombre} hace un sonido."

    def comer(self, alimento: str) -> str:
        self._energia += 20
        return f"{self.nombre} come {alimento}. Energía: {self._energia}"

    def __repr__(self) -> str:
        return f"Animal(nombre='{self.nombre}', especie='{self.especie}')"


class Perro(Animal):
    """Subclase que extiende Animal."""

    def __init__(self, nombre: str, raza: str) -> None:
        super().__init__(nombre, especie="Canino")  # llama al padre
        self.raza = raza

    def hablar(self) -> str:  # override
        return f"{self.nombre} dice: ¡Guau!"

    def buscar(self, objeto: str) -> str:
        """Método propio de Perro."""
        return f"{self.nombre} busca el {objeto}. ¡Guau!"


class Gato(Animal):
    """Otra subclase."""

    def __init__(self, nombre: str, indoor: bool = True) -> None:
        super().__init__(nombre, especie="Felino")
        self.indoor = indoor

    def hablar(self) -> str:  # override
        return f"{self.nombre} dice: ¡Miau!"

    def ronronear(self) -> str:
        return f"{self.nombre} ronronea... purrrrr"

In [ ]:
# Crear instancias
rex = Perro("Rex", "Pastor Alemán")
miau = Gato("Michi", indoor=False)

# Métodos heredados
print(rex.comer("croquetas"))
print(miau.comer("atún"))
print()

# Métodos sobreescritos (override)
print(rex.hablar())
print(miau.hablar())
print()

# Métodos propios
print(rex.buscar("palo"))
print(miau.ronronear())

### 4.2 `isinstance` y `issubclass`

Estas funciones permiten verificar relaciones de herencia en tiempo de ejecución.

In [ ]:
# isinstance: ¿es esta instancia de esta clase (o subclase)?
print(f"rex es Perro? {isinstance(rex, Perro)}")
print(f"rex es Animal? {isinstance(rex, Animal)}")
print(f"rex es Gato? {isinstance(rex, Gato)}")
print()

# issubclass: ¿es esta clase subclase de otra?
print(f"Perro es subclase de Animal? {issubclass(Perro, Animal)}")
print(f"Gato es subclase de Animal? {issubclass(Gato, Animal)}")
print(f"Animal es subclase de Perro? {issubclass(Animal, Perro)}")
print()

# Patrón útil: polimorfismo con isinstance
animales: list[Animal] = [rex, miau, Animal("Búho", "Ave")]
for animal in animales:
    if isinstance(animal, Perro):
        print(f"  🐕 {animal.nombre} → {animal.hablar()} (raza: {animal.raza})")
    elif isinstance(animal, Gato):
        print(f"  🐈 {animal.nombre} → {animal.hablar()}")
    else:
        print(f"  🐾 {animal.nombre} → {animal.hablar()}")

## 5. Herencia Múltiple y MRO

### 5.1 Herencia múltiple

Una clase puede heredar de **múltiples padres**. Esto es poderoso pero requiere entender cómo Python resuelve las llamadas.

In [ ]:
class Volador:
    """Mixin: capacidad de volar."""

    def volar(self) -> str:
        return f"{getattr(self, 'nombre', 'Anonimo')} está volando."


class Nadador:
    """Mixin: capacidad de nadar."""

    def nadar(self) -> str:
        return f"{getattr(self, 'nombre', 'Anonimo')} está nadando."


class Patito(Volador, Nadador):
    """Un patito que puede volar y nadar — herencia múltiple."""

    def __init__(self, nombre: str) -> None:
        self.nombre = nombre

    def hablar(self) -> str:
        return f"{self.nombre} dice: ¡Cuac!"


pato = Patito("Pepe")
print(pato.hablar())
print(pato.volar())
print(pato.nadar())

### 5.2 El problema del diamante y C3 Linearization

Cuando múltiples clases heredan de una misma superclase, se crea un **diamante**. Python lo resuelve con **C3 Linearization**, un algoritmo que define un orden determinista (MRO - Method Resolution Order).

```
        ┌─────────┐
        │   A     │  ← clase base
        └────┬────┘
         ┌───┴───┐
    ┌────┴──┐ ┌──┴────┐
    │   B   │ │   C   │  ← heredan de A
    └───┬───┘ └───┬───┘
        └────┬────┘
         ┌───┴───┐
    ┌────┴──────┐│
    │     D     ││  ← hereda de B y C (diamante)
    └───────────┘│

    MRO de D: D → B → C → A → object
    (izquierda a derecha, profundidad primero, sin duplicados)
```

In [ ]:
class A:
    """Clase base del diamante."""

    def metodo(self) -> str:
        return "Soy A"


class B(A):
    """Hereda de A."""

    def metodo(self) -> str:
        return "Soy B"


class C(A):
    """Hereda de A."""

    def metodo(self) -> str:
        return "Soy C"


class D(B, C):
    """Hereda de B y C — diamante."""

    pass


d = D()
print(f"d.metodo() → {d.metodo()}")
print(f"MRO: {[cls.__name__ for cls in D.__mro__]}")
print(f"MRO directo: {D.mro()}")

### 5.3 `super()` en herencia múltiple

`super()` **no siempre llama al padre directo**. Sigue el MRO. Esto es crítico para entender:

In [ ]:
class Base:
    def __init__(self) -> None:
        print("  Base.__init__")


class Izquierda(Base):
    def __init__(self) -> None:
        print("  Izquierda.__init__")
        super().__init__()


class Derecha(Base):
    def __init__(self) -> None:
        print("  Derecha.__init__")
        super().__init__()


class Centro(Izquierda, Derecha):
    def __init__(self) -> None:
        print("  Centro.__init__")
        super().__init__()


print("Construyendo Centro (muestra el orden C3):")
c = Centro()
print(f"\nMRO: {[cls.__name__ for cls in Centro.__mro__]}")

print("\nNota: Derecha.__init__ llama super() que va a Base, no a Izquierda.")
print("El super() Sigue el MRO, NO el padre textual.")

In [ ]:
# Diagrama visual del MRO
print("=" * 60)
print("         MRO de Centro (C3 Linearization)")
print("=" * 60)
print()
print("  MRO: Centro → Izquierda → Derecha → Base → object")
print()
print("  Flujo de super().__init__():")
print()
print("  Centro.__init__")
print("       │")
print("       ▼  super() → siguiente en MRO")
print("  Izquierda.__init__")
print("       │")
print("       ▼  super() → siguiente en MRO")
print("  Derecha.__init__")
print("       │")
print("       ▼  super() → siguiente en MRO")
print("  Base.__init__")
print("       │")
print("       ▼  super() → object")
print("  object.__init__")

### 5.4 Diagrama completo de jerarquías

```
                    ┌──────────────────┐
                    │     object       │
                    └────────┬─────────┘
                             │
                    ┌────────┴─────────┐
                    │     Animal       │
                    └────────┬─────────┘
              ┌──────────────┼──────────────┐
              │              │              │
     ┌────────┴───┐  ┌──────┴──────┐  ┌───┴────────┐
     │   Perro    │  │    Gato     │  │    Ave     │
     └────────┬───┘  └─────────────┘  └───┬────────┘
              │                           │
              │              ┌────────────┼──────────┐
              │              │            │          │
              │        ┌─────┴──┐  ┌──────┴───┐  ┌──┴───────┐
              │        │ Agila  │  │  Loro    │  │  Pato    │
              │        └────────┘  └──────────┘  │(+Volador │
              │                                  │ +Nadador)│
              │                                  └──────────┘

  ┌──────────────────────────────────────────────────────┐
  │  Mixins (herencia múltiple):                        │
  │  Volador, Nadador → se "pegan" a la clase que       │
  │  necesite esa capacidad, sin crear jerarquía rígida  │
  └──────────────────────────────────────────────────────┘
```

## 6. Composición vs Herencia

### 6.1 La regla de oro: "Favor composition over inheritance"

- **Herencia** = relación **"es un"**. Un `Perro` *es un* `Animal`.
- **Composición** = relación **"tiene un"**. Un `Auto` *tiene un* `Motor`.

**Cuándo usar herencia**:
- La relación es genuinamente "es un" y es estable.
- Necesitas polimorfismo natural.

**Cuándo usar composición**:
- La relación es "tiene un", "usa un" o "contiene".
- Quieres cambiar comportamiento en tiempo de ejecución.
- La herencia crea acoplamiento excesivo.
- Necesitas reutilizar comportamiento sin crear jerarquías profundas.

In [ ]:
# MAL EJEMPLO: Herencia cuando debería ser composición
class MySQLDatabase:
    def __init__(self, host: str) -> None:
        self.host = host

    def conectar(self) -> str:
        return f"Conectado a MySQL en {self.host}"


class UsuarioConMySQL(MySQLDatabase):  # ¿Un usuario "es una" base de datos? NO.
    def __init__(self, nombre: str, host: str) -> None:
        super().__init__(host)
        self.nombre = nombre

    def crear(self) -> str:
        return f"Creando usuario {self.nombre} en {self.conectar()}"


# Este diseño es FRÁGIL: si cambias a PostgreSQL, tienes que reescribir UsuarioConMySQL.
print("MAL DISEÑO (herencia):")
u = UsuarioConMySQL("admin", "localhost")
print(f"  {u.crear()}")

In [ ]:
from abc import ABC, abstractmethod


# BUEN EJEMPLO: Composición con inyección de dependencias
class BaseDatos(ABC):
    """Interfaz abstracta para cualquier base de datos."""

    @abstractmethod
    def conectar(self) -> str: ...

    @abstractmethod
    def ejecutar(self, query: str) -> str: ...


class MySQL(BaseDatos):
    def __init__(self, host: str) -> None:
        self.host = host

    def conectar(self) -> str:
        return f"MySQL@{self.host}"

    def ejecutar(self, query: str) -> str:
        return f"[MySQL] {query}"


class SQLite(BaseDatos):
    def __init__(self, archivo: str) -> None:
        self.archivo = archivo

    def conectar(self) -> str:
        return f"SQLite@{self.archivo}"

    def ejecutar(self, query: str) -> str:
        return f"[SQLite] {query}"


class Usuario:
    """Usuario usa una base de datos (composición)."""

    def __init__(self, nombre: str, db: BaseDatos) -> None:
        self.nombre = nombre
        self._db = db  # inyección de dependencia

    def crear(self) -> str:
        return f"{self.nombre} creado en {self._db.conectar()}"

    def guardar(self) -> str:
        return self._db.ejecutar(f"INSERT INTO usuarios VALUES('{self.nombre}')")


# Ahora puedes intercambiar la base de datos sin modificar Usuario
usuario_mysql = Usuario("Ana", MySQL("localhost"))
usuario_sqlite = Usuario("Luis", SQLite("mi_base.db"))

print("BUEN DISEÑO (composición):")
print(f"  {usuario_mysql.crear()}")
print(f"  {usuario_mysql.guardar()}")
print(f"  {usuario_sqlite.crear()}")
print(f"  {usuario_sqlite.guardar()}")

### 6.2 Comparación directa

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Criterio              │  Herencia              │  Composición         │
├────────────────────────┼────────────────────────┼──────────────────────┤
│  Relación              │  "es un"               │  "tiene un"          │
│  Acoplamiento          │  Fuerte                │  Débil                │
│  Cambio en runtime     │  No posible            │  Sí posible          │
│  Reutilización         │  Jerárquica            │  Flexible             │
│  Complejidad           │  Aumenta con niveles   │  Se mantiene plana   │
│  Testing               │  Más difícil           │  Más fácil (mocks)   │
│  Ejemplo               │  Gato → Animal         │  Auto tiene Motor    │
│  Python lo resuelve    │  MRO / super()         │  Delegación / ABCs   │
└─────────────────────────────────────────────────────────────────────────┘

  REGLA PRÁCTICA:
  ┌─────────────────────────────────────────────────────────────────┐
  │  ¿Puedo decir "X es un Y" sin sonar absurdo?                   │
  │    SÍ  → Herencia                                              │
  │    NO  → Composición                                           │
  │                                                                 │
  │  ¿Necesito cambiar comportamiento dinámicamente?                │
  │    SÍ  → Composición                                           │
  │    NO  → Ambas funcionan, pero composición es más flexible     │
  └─────────────────────────────────────────────────────────────────┘
```

## 7. Tabla de Referencia: `@property` / `@classmethod` / `@staticmethod`

```
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                        │  @property          │  @classmethod      │  @staticmethod       │
├────────────────────────┼─────────────────────┼────────────────────┼──────────────────────┤
│ Primer argumento      │  self (implícito)   │  cls (la clase)    │  Ninguno             │
│ Accede a instancia?   │  Sí                 │  No directamente   │  No                  │
│ Accede a clase?       │  Sí vía self.__clase│  Sí (cls)          │  No                  │
│ Propósito principal   │  Get/set con lógica │  Factory methods   │  Función utilitaria  │
│ Sintaxis de uso       │  obj.propiedad      │  Clase.metodo()    │  Clase.metodo()      │
│ Puede heredarse?      │  Sí (override)      │  Sí (recibe subclase│  No (es independiente│
│ Ejemplo típico        │  Validación de datos│  from_string()     │  es_par()            │
└─────────────────────────────────────────────┴────────────────────┴──────────────────────┘
```

## 8. Ejercicios

### Ejercicio 1 (Guiado): Sistema de descendencia de animales

Crea una jerarquía completa con `Animal` como base, `Mamifero` y `Reptil` como subclases, y luego `Perro`, `Gato`, `Serpiente` y `Lagarto`. Implementa `hablar()`, un mixin `Domestico` y verifica con `isinstance`.

In [ ]:
class Animal:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre

    def hablar(self) -> str:
        raise NotImplementedError("Subdebe implementar hablar()")

    def __repr__(self) -> str:
        return f"{type(self).__name__}(nombre='{self.nombre}')"


class Domestico:
    """Mixin: indica que el animal es domesticable."""
    es_domestico: bool = True


class Mamifero(Animal):
    tipo: str = "Mamífero"


class Reptil(Animal):
    tipo: str = "Reptil"


class Perro(Mamifero, Domestico):
    def hablar(self) -> str:
        return f"{self.nombre}: ¡Guau!"


class Gato(Mamifero, Domestico):
    def hablar(self) -> str:
        return f"{self.nombre}: ¡Miau!"


class Serpiente(Reptil):
    def hablar(self) -> str:
        return f"{self.nombre}: ¡Ssssss!"


class Lagarto(Reptil):
    def hablar(self) -> str:
        return f"{self.nombre}: ...silencio"


# Verificación
mascotas: list[Animal] = [Perro("Rex"), Gato("Michi"), Serpiente("Kaa"), Lagarto("Iggy")]

for animal in mascotas:
    domestico = "(doméstico)" if isinstance(animal, Domestico) else "(salvaje)"
    print(f"{animal!r} {domestico} → {animal.hablar()}")

print(f"\nPerro es subclase de Mamifero? {issubclass(Perro, Mamifero)}")
print(f"Gato es subclase de Domestico? {issubclass(Gato, Domestico)}")
print(f"Serpiente es subclase de Domestico? {issubclass(Serpiente, Domestico)}")

### Ejercicio 2 (Guiado): Factory method con herencia

Crea una clase `Figura` con un factory method `desde_dict()` y subclases `Circulo`, `Rectangulo` y `Triangulo` que calculen su área.

In [ ]:
import math
from abc import ABC, abstractmethod


class Figura(ABC):
    """Figura geométrica abstracta con factory method."""

    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimetro(self) -> float: ...

    @classmethod
    def desde_dict(cls, datos: dict) -> "Figura":
        """Factory method: crea la figura correcta desde un diccionario."""
        tipo = datos["tipo"]
        mapa: dict[str, type[Figura]] = {
            "circulo": Circulo,
            "rectangulo": Rectangulo,
            "triangulo": Triangulo,
        }
        if tipo not in mapa:
            raise ValueError(f"Tipo desconocido: {tipo}")
        clase = mapa[tipo]
        params = {k: v for k, v in datos.items() if k != "tipo"}
        return clase(**params)

    def __repr__(self) -> str:
        return f"{type(self).__name__}(area={self.area():.2f}, perimetro={self.perimetro():.2f})"


class Circulo(Figura):
    def __init__(self, radio: float) -> None:
        self.radio = radio

    def area(self) -> float:
        return math.pi * self.radio ** 2

    def perimetro(self) -> float:
        return 2 * math.pi * self.radio


class Rectangulo(Figura):
    def __init__(self, base: float, altura: float) -> None:
        self.base = base
        self.altura = altura

    def area(self) -> float:
        return self.base * self.altura

    def perimetro(self) -> float:
        return 2 * (self.base + self.altura)


class Triangulo(Figura):
    def __init__(self, a: float, b: float, c: float) -> None:
        self.a, self.b, self.c = a, b, c

    def area(self) -> float:
        s = (self.a + self.b + self.c) / 2  # semiperímetro
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))

    def perimetro(self) -> float:
        return self.a + self.b + self.c


# Crear figuras desde diccionarios
figuras_data = [
    {"tipo": "circulo", "radio": 5},
    {"tipo": "rectangulo", "base": 4, "altura": 7},
    {"tipo": "triangulo", "a": 3, "b": 4, "c": 5},
]

figuras = [Figura.desde_dict(d) for d in figuras_data]

for fig in figuras:
    print(f"{fig}")

### Ejercicio 3 (Guiado): Composición con `@property`

Crea un sistema de empleados donde `Departamento` contiene múltiples `Empleado` (composición), y los salarios se controlan con `@property` con validación.

In [ ]:
class Empleado:
    """Empleado con salario encapsulado."""

    def __init__(self, nombre: str, cargo: str, salario: float) -> None:
        self.nombre = nombre
        self.cargo = cargo
        self._salario = 0.0
        self.salario = salario  # usa el setter para validar

    @property
    def salario(self) -> float:
        return self._salario

    @salario.setter
    def salario(self, valor: float) -> None:
        if valor < 0:
            raise ValueError(f"Salario no puede ser negativo: {valor}")
        self._salario = valor

    def __repr__(self) -> str:
        return f"{self.nombre} ({self.cargo}) - ${self._salario:,.2f}"


class Departamento:
    """Departamento compuesto por empleados (composición)."""

    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self._empleados: list[Empleado] = []

    def contratar(self, empleado: Empleado) -> None:
        self._empleados.append(empleado)

    def despedir(self, nombre: str) -> Empleado | None:
        for i, emp in enumerate(self._empleados):
            if emp.nombre == nombre:
                return self._empleados.pop(i)
        return None

    @property
    def total_salarios(self) -> float:
        return sum(e.salario for e in self._empleados)

    @property
    def cantidad(self) -> int:
        return len(self._empleados)

    def __repr__(self) -> str:
        return f"Departamento('{self.nombre}', empleados={self.cantidad}, costo_total=${self.total_salarios:,.2f})"


# Crear departamento
tech = Departamento("Tecnología")
tech.contratar(Empleado("Ana", "Lead Dev", 85000))
tech.contratar(Empleado("Carlos", "Backend Dev", 72000))
tech.contratar(Empleado("Laura", "Frontend Dev", 68000))

print(tech)
print()
for emp in tech._empleados:
    print(f"  {emp}")
print(f"\nCosto total anual: ${tech.total_salarios:,.2f}")

### Ejercicio 4 (Independiente): Sistema Bancario con POO

Implementa un sistema bancario que demuestre todos los conceptos vistos. Sigue las indicaciones paso a paso:

In [ ]:
from datetime import date, datetime
from abc import ABC, abstractmethod


# ─── PASO 1: Clase abstracta base ────────────────────────────────
class CuentaBancaria(ABC):
    """Cuenta bancaria abstracta con encapsulamiento."""

    _contador: int = 0

    def __init__(self, titular: str, saldo: float = 0.0) -> None:
        CuentaBancaria._contador += 1
        self._numero = f"CTA-{CuentaBancaria._contador:06d}"
        self._titular = titular
        self._saldo = saldo
        self._historial: list[str] = []
        self._registrar(f"Cuenta creada con saldo ${saldo:,.2f}")

    @property
    def titular(self) -> str:
        return self._titular

    @property
    def saldo(self) -> float:
        return self._saldo

    @property
    def numero(self) -> str:
        return self._numero

    @property
    def historial(self) -> list[str]:
        return self._historial.copy()

    def depositar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto debe ser positivo")
        self._saldo += monto
        self._registrar(f"Depósito +${monto:,.2f}")

    def retirar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto debe ser positivo")
        if monto > self._saldo:
            raise ValueError(f"Saldo insuficiente: ${self._saldo:,.2f} < ${monto:,.2f}")
        self._saldo -= monto
        self._registrar(f"Retiro -${monto:,.2f}")

    @abstractmethod
    def tasa_interes(self) -> float:
        """Cada subclase define su propia tasa."""

    def aplicar_interes(self) -> float:
        interes = self._saldo * self.tasa_interes()
        self._saldo += interes
        self._registrar(f"Interés +${interes:,.2f} ({self.tasa_interes()*100:.1f}%)")
        return interes

    def _registrar(self, mensaje: str) -> None:
        fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
        self._historial.append(f"[{fecha}] {mensaje}")

    @classmethod
    def desde_dict(cls, data: dict) -> "CuentaBancaria":
        tipo = data["tipo"]
        mapa: dict[str, type[CuentaBancaria]] = {
            "ahorro": CuentaAhorro,
            "corriente": CuentaCorriente,
        }
        if tipo not in mapa:
            raise ValueError(f"Tipo no soportado: {tipo}")
        return mapa[tipo](data["titular"], data.get("saldo", 0))

    def __repr__(self) -> str:
        return f"{type(self).__name__}(numero='{self._numero}', titular='{self._titular}', saldo=${self._saldo:,.2f})"


# ─── PASO 2: Subclases concretas ─────────────────────────────────
class CuentaAhorro(CuentaBancaria):
    """Cuenta de ahorro con tasa del 3%."""

    TASA: float = 0.03

    def tasa_interes(self) -> float:
        return self.TASA


class CuentaCorriente(CuentaBancaria):
    """Cuenta corriente con sobregiro permitido."""

    TASA: float = 0.01
    LIMITE_SOBREGIRO: float = 500.0

    def tasa_interes(self) -> float:
        return self.TASA

    def retirar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto debe ser positivo")
        if monto > self._saldo + self.LIMITE_SOBREGIRO:
            raise ValueError(f"Excede límite de sobregiro (${self.LIMITE_SOBREGIRO:,.2f})")
        self._saldo -= monto
        self._registrar(f"Retiro -${monto:,.2f}")


# ─── PASO 3: Usar el sistema ─────────────────────────────────────
ahorro = CuentaAhorro("Ana García", 10000)
corriente = CuentaCorriente("Luis Pérez", 3000)

print("=== Cuentas creadas ===")
print(f"  {ahorro}")
print(f"  {corriente}")
print()

# Operaciones
ahorro.depositar(5000)
ahorro.retirar(2000)
interes_ahorro = ahorro.aplicar_interes()

print("=== Operaciones en CuentaAhorro ===")
print(f"  Saldo final: ${ahorro.saldo:,.2f}")
print(f"  Interés generado: ${interes_ahorro:,.2f}")
print()

# Sobregiro en cuenta corriente
corriente.retirar(3400)  # se va a -400 (dentro del límite)
print("=== Operaciones en CuentaCorriente ===")
print(f"  Saldo después de sobregiro: ${corriente.saldo:,.2f}")

try:
    corriente.retirar(200)  # excede el límite
except ValueError as e:
    print(f"  Error: {e}")
print()

# Factory method
data = {"tipo": "ahorro", "titular": "Sofía López", "saldo": 7500}
nueva = CuentaBancaria.desde_dict(data)
print(f"=== Creada vía factory method ===")
print(f"  {nueva}")
print(f"  isinstance(nueva, CuentaAhorro)? {isinstance(nueva, CuentaAhorro)}")
print(f"  isinstance(nueva, CuentaBancaria)? {isinstance(nueva, CuentaBancaria)}")
print()

# Historial
print("=== Historial de Ana ===")
for registro in ahorro.historial:
    print(f"  {registro}")

## Resumen

```
┌──────────────────────────────────────────────────────────────────────────┐
│                    POO en Python — Mapa de Conceptos                    │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  FUNDAMENTOS                                                           │
│  ├── class + __init__(self) → crear clases y objetos                    │
│  ├── self → referencia al objeto actual                                │
│  ├── Type hints → documentar tipos en parámetros y retorno             │
│  ├── Sobrecarga NO existe → usar @overload + isinstance                 │
│  └── Dunder methods → integrar con operadores (+, ==, repr, etc.)       │
│                                                                        │
│  ENCAPSULAMIENTO                                                       │
│  ├── @property → getter/setter/deleter con validación                  │
│  ├── _atributo → convención "protegido"                                │
│  └── __atributo → name mangling (_Clase__atributo)                     │
│                                                                        │
│  MÉTODOS ESPECIALES                                                   │
│  ├── @classmethod → factory methods (recibe cls)                       │
│  └── @staticmethod → funciones utilitarias (sin self ni cls)           │
│                                                                        │
│  HERENCIA                                                              │
│  ├── Subclase( Superclase ) → hereda atributos y métodos               │
│  ├── super() → llama al padre (sigue MRO)                              │
│  ├── isinstance / issubclass → verificar relaciones                    │
│  └── Herencia múltiple → MRO vía C3 linearization                     │
│                                                                        │
│  COMPOSICIÓN vs HERENCIA                                               │
│  ├── "Es un" → herencia (Gato es Animal)                               │
│  ├── "Tiene un" → composición (Auto tiene Motor)                       │
│  └── Favor composition over inheritance                                │
│                                                                        │
└──────────────────────────────────────────────────────────────────────────┘
```